# Flash Floods KY Python Scripts

## Script #1: Moon Sun Calculations

In [141]:
# Import Libraries and Dependencies
from datetime import datetime
from pathlib import Path
import math
import ephem
import pandas as pd

In [142]:
# Read dataset into dataframe, then inspect it
df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

df.head()

,event_id,county_name,begin_location,begin_date,begin_time,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,injuries_indirect,...,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,0,0,0,0,0,...,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,0,0,20000,0,0,...,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0,0,30000,0,0,...,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0,0,30000,0,0,...,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0,0,0,0,0,...,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [143]:
# Create Astronomy Calculation Function
def calculate_local_astro_data(row: pd.Series) -> pd.Series:
    """Calculate sun and moon data for one row of the dataset."""
    
    try:
        # Get the event date
        date_obj = datetime.strptime(row["begin_date"], "%Y-%m-%d")
        
        # Create an observer using the event's coordinates
        observer = ephem.Observer()
        observer.lat = str(row["begin_lat"])
        observer.lon = str(row["begin_lon"])
        observer.date = date_obj
        
        # --------------------------------------------------
        # 1. SUN CALCULATIONS
        # --------------------------------------------------
        
        sun = ephem.Sun()
        sun.compute(observer)
        
        # Calculate sun altitude and azimuth
        sun_alt_deg = math.degrees(sun.alt)
        sun_az_deg = math.degrees(sun.az)
        
        # Reset observer date to the beginning of the day
        # to calculate sunrise and sunset
        observer.date = date_obj.date()
        
        try:
            sunrise = observer.next_rising(sun)
            sunrise_str = sunrise.datetime().strftime("%Y-%m-%d %H:%M:%S")
        except (ephem.CircumpolarError, ephem.AlwaysUpError):
            sunrise_str = "Sun always up"
        except ephem.NeverUpError:
            sunrise_str = "Sun never rises"
        
        try:
            sunset = observer.next_setting(sun)
            sunset_str = sunset.datetime().strftime("%Y-%m-%d %H:%M:%S")
        except (ephem.CircumpolarError, ephem.AlwaysUpError):
            sunset_str = "Sun always up"
        except ephem.NeverUpError:
            sunset_str = "Sun never sets"
        
        # --------------------------------------------------
        # 2. MOON CALCULATIONS
        # --------------------------------------------------
        
        # Reset observer to the event's exact date and time
        observer.date = date_obj
        
        moon = ephem.Moon()
        moon.compute(observer)
        
        # Calculate moon altitude and azimuth
        moon_alt_deg = math.degrees(moon.alt)
        moon_az_deg = math.degrees(moon.az)
        
        # Calculate moon illumination
        illumination = moon.phase / 100.0
        
        # Determine the moon's position within its lunar cycle
        prev_new = ephem.previous_new_moon(observer.date)
        next_new = ephem.next_new_moon(observer.date)
        
        lunation_age = (
            (observer.date - prev_new)
            * 29.53
            / (next_new - prev_new)
        )
        
        # Determine moon phase
        if illumination < 0.03:
            phase_name = "New Moon"
        
        elif illumination > 0.97:
            phase_name = "Full Moon"
        
        elif lunation_age < 14.77:
            if illumination < 0.45:
                phase_name = "Waxing Crescent"
            elif illumination < 0.55:
                phase_name = "First Quarter"
            else:
                phase_name = "Waxing Gibbous"
        
        else:
            if illumination > 0.55:
                phase_name = "Waning Gibbous"
            elif illumination > 0.45:
                phase_name = "Third Quarter"
            else:
                phase_name = "Waning Crescent"
        
        # Return the calculated astronomical data
        return pd.Series({
            "sun_altitude_deg": round(sun_alt_deg, 2),
            "sun_azimuth_deg": round(sun_az_deg, 2),
            "sunrise_utc": sunrise_str,
            "sunset_utc": sunset_str,
            "moon_altitude_deg": round(moon_alt_deg, 2),
            "moon_azimuth_deg": round(moon_az_deg, 2),
            "moon_phase_name": phase_name,
            "moon_illumination_pct": round(illumination * 100, 2)
        })
    
    except Exception as e:
        # Return error information if a calculation fails
        return pd.Series({
            "sun_altitude_deg": None,
            "sun_azimuth_deg": None,
            "sunrise_utc": f"Error: {str(e)}",
            "sunset_utc": f"Error: {str(e)}",
            "moon_altitude_deg": None,
            "moon_azimuth_deg": None,
            "moon_phase_name": f"Error: {str(e)}",
            "moon_illumination_pct": None
        })

In [144]:
# Run astronomical calculations and display results
astro_data = df.apply(calculate_local_astro_data, axis=1)

astro_data.head()

,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
0,0.85,276.15,2015-04-03 11:24:48,2015-04-03 00:06:46,14.25,101.85,Full Moon,97.95
1,0.88,276.12,2015-04-03 11:24:57,2015-04-03 00:06:56,14.21,101.82,Full Moon,97.95
2,0.31,276.63,2015-04-03 11:21:47,2015-04-03 00:03:35,14.87,102.33,Full Moon,97.95
3,0.65,276.33,2015-04-03 11:23:41,2015-04-03 00:05:35,14.48,102.02,Full Moon,97.95
4,0.94,276.07,2015-04-03 11:25:13,2015-04-03 00:07:18,14.14,101.79,Full Moon,97.95


In [145]:
# Combine event_id with the calculated astronomy data
output_df = pd.concat(
    [df[["event_id"]], astro_data],
    axis=1
)

output_df.head()

,event_id,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
0,564702,0.85,276.15,2015-04-03 11:24:48,2015-04-03 00:06:46,14.25,101.85,Full Moon,97.95
1,564703,0.88,276.12,2015-04-03 11:24:57,2015-04-03 00:06:56,14.21,101.82,Full Moon,97.95
2,564704,0.31,276.63,2015-04-03 11:21:47,2015-04-03 00:03:35,14.87,102.33,Full Moon,97.95
3,564706,0.65,276.33,2015-04-03 11:23:41,2015-04-03 00:05:35,14.48,102.02,Full Moon,97.95
4,564705,0.94,276.07,2015-04-03 11:25:13,2015-04-03 00:07:18,14.14,101.79,Full Moon,97.95


In [146]:
# Save dataframe as CSV
output_df.to_csv("../data/processed/flash_floods_ky_moon_sun_data.csv", index=False)

## Script #2: Chi-Square Goodness of Fit

In [147]:
# Import Libraries and Dependencies
from pathlib import Path
import pandas as pd
from scipy.stats import chisquare

In [148]:
# Read dataset into dataframe
df = pd.read_csv("../data/processed/flash_floods_ky_moon_sun_data.csv")

df.head()

,event_id,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
0,564702,0.85,276.15,2015-04-03 11:24:48,2015-04-03 00:06:46,14.25,101.85,Full Moon,97.95
1,564703,0.88,276.12,2015-04-03 11:24:57,2015-04-03 00:06:56,14.21,101.82,Full Moon,97.95
2,564704,0.31,276.63,2015-04-03 11:21:47,2015-04-03 00:03:35,14.87,102.33,Full Moon,97.95
3,564706,0.65,276.33,2015-04-03 11:23:41,2015-04-03 00:05:35,14.48,102.02,Full Moon,97.95
4,564705,0.94,276.07,2015-04-03 11:25:13,2015-04-03 00:07:18,14.14,101.79,Full Moon,97.95


In [149]:
# Create bin boundaries 
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

In [150]:
# Create bin labels
labels = [
    "0-10%", 
    "10-20%", 
    "20-30%", 
    "30-40%", 
    "40-50%", 
    "50-60%", 
    "60-70%", 
    "70-80%", 
    "80-90%", 
    "90-100%"
]

In [151]:
# Assign each flash flooding event to a moon illumination bin
df["illumination_bin"] = pd.cut(
    df["moon_illumination_pct"], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
)

df[["moon_illumination_pct", "illumination_bin"]].head(20)

,moon_illumination_pct,illumination_bin
0,97.95,90-100%
1,97.95,90-100%
2,97.95,90-100%
3,97.95,90-100%
4,97.95,90-100%
5,97.95,90-100%
6,97.95,90-100%
7,97.95,90-100%
8,97.95,90-100%
9,97.95,90-100%


In [152]:
# Count observed flash flooding events
observed = df["illumination_bin"].value_counts().sort_index()

observed

illumination_bin
0-10%      453
10-20%     121
20-30%     108
30-40%      92
40-50%     114
50-60%     125
60-70%      67
70-80%      62
80-90%     192
90-100%    423
Name: count, dtype: int64

In [153]:
# Calculate expected flash flooding event counts
expected = [len(df) / len(observed)] * len(observed)

expected

[175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7]

In [154]:
# Run Chi-Square Goodness of Fit test and display the results
chi2, p_value = chisquare(
    f_obs=observed, 
    f_exp=expected
)

print("Observed Counts:")
print(observed)

print("\nExpected Counts:")
print(expected)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6f}")

Observed Counts:
illumination_bin
0-10%      453
10-20%     121
20-30%     108
30-40%      92
40-50%     114
50-60%     125
60-70%      67
70-80%      62
80-90%     192
90-100%    423
Name: count, dtype: int64

Expected Counts:
[175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7]

Chi-Square Statistic: 1047.3540
P-Value: 0.000000


## Script #3: Oceanic Nino Index 

In [155]:
# Import Libraries and Dependencies
import io
from pathlib import Path
import numpy as np
import pandas as pd

In [156]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,injuries_indirect,...,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,0,0,0,0,0,...,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,0,0,20000,0,0,...,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0,0,30000,0,0,...,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0,0,30000,0,0,...,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0,0,0,0,0,...,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [157]:
# Read the ONI text file
with open("../data/raw/oni_backup.txt", "r", encoding="utf-8") as f: oni_text = f.read()

print(oni_text[:1000])

 SEAS  YR   TOTAL   ANOM
  DJF 1950  24.72  -1.53
  JFM 1950  25.17  -1.34
  FMA 1950  25.75  -1.16
  MAM 1950  26.12  -1.18
  AMJ 1950  26.32  -1.07
  MJJ 1950  26.31  -0.85
  JJA 1950  26.21  -0.54
  JAS 1950  25.96  -0.42
  ASO 1950  25.76  -0.39
  SON 1950  25.63  -0.44
  OND 1950  25.48  -0.60
  NDJ 1950  25.34  -0.80
  DJF 1951  25.42  -0.82
  JFM 1951  25.96  -0.54
  FMA 1951  26.74  -0.17
  MAM 1951  27.48   0.18
  AMJ 1951  27.75   0.36
  MJJ 1951  27.75   0.58
  JJA 1951  27.44   0.70
  JAS 1951  27.28   0.89
  ASO 1951  27.14   0.99
  SON 1951  27.22   1.15
  OND 1951  27.12   1.04
  NDJ 1951  26.95   0.81
  DJF 1952  26.78   0.53
  JFM 1952  26.87   0.37
  FMA 1952  27.25   0.34
  MAM 1952  27.60   0.29
  AMJ 1952  27.59   0.20
  MJJ 1952  27.17   0.00
  JJA 1952  26.67  -0.08
  JAS 1952  26.39   0.00
  ASO 1952  26.30   0.15
  SON 1952  26.17   0.10
  OND 1952  26.13   0.04
  NDJ 1952  26.29   0.15
  DJF 1953  26.65   0.40
  JFM 1953  27.10   0.60
  FMA 1953  27.53   0.63


In [158]:
# Parse ONI text into a dataframe, then check results
oni_df = pd.read_csv(
    io.StringIO(oni_text.strip()),
    sep=r"\s+",
    dtype={"YR": int, "ANOM": float}
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM
0,DJF,1950,24.72,-1.53
1,JFM,1950,25.17,-1.34
2,FMA,1950,25.75,-1.16
3,MAM,1950,26.12,-1.18
4,AMJ,1950,26.32,-1.07


In [159]:
# Map ONI seasons to a central calendar month, then check results
season_to_month = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}

oni_df["month"] = oni_df["SEAS"].map(season_to_month)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month
0,DJF,1950,24.72,-1.53,1
1,JFM,1950,25.17,-1.34,2
2,FMA,1950,25.75,-1.16,3
3,MAM,1950,26.12,-1.18,4
4,AMJ,1950,26.32,-1.07,5


In [160]:
# Create a date column using the year and month, then check results
oni_df["date"] = pd.to_datetime(
    dict(
        year=oni_df["YR"],
        month=oni_df["month"],
        day=1
    )
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month,date
0,DJF,1950,24.72,-1.53,1,1950-01-01
1,JFM,1950,25.17,-1.34,2,1950-02-01
2,FMA,1950,25.75,-1.16,3,1950-03-01
3,MAM,1950,26.12,-1.18,4,1950-04-01
4,AMJ,1950,26.32,-1.07,5,1950-05-01


In [161]:
# Classify ENSO phase based on the ONI anomaly, then check results
oni_df["enso_phase"] = np.select(
    [
        oni_df["ANOM"] >= 0.5,
        oni_df["ANOM"] <= -0.5
    ],
    [
        "El Nino",
        "La Nina"
    ],
    default="Neutral"
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month,date,enso_phase
0,DJF,1950,24.72,-1.53,1,1950-01-01,La Nina
1,JFM,1950,25.17,-1.34,2,1950-02-01,La Nina
2,FMA,1950,25.75,-1.16,3,1950-03-01,La Nina
3,MAM,1950,26.12,-1.18,4,1950-04-01,La Nina
4,AMJ,1950,26.32,-1.07,5,1950-05-01,La Nina


In [ ]:
# Select the columns needed for the analysis and rename them for clarity, then check results
oni_df = oni_df[
    ["date", "SEAS", "ANOM", "enso_phase"]
].rename(
    columns={
        "SEAS": "oni_season",
        "ANOM": "oni_anomaly"
    }
)

oni_df.head()

,date,oni_season,oni_anomaly,enso_phase
0,1950-01-01,DJF,-1.53,La Nina
1,1950-02-01,JFM,-1.34,La Nina
2,1950-03-01,FMA,-1.16,La Nina
3,1950-04-01,MAM,-1.18,La Nina
4,1950-05-01,AMJ,-1.07,La Nina


In [ ]:
# Sort ONI data chronologically, then check results
oni_df = oni_df.sort_values("date").reset_index(drop=True)

oni_df.head()

,date,oni_season,oni_anomaly,enso_phase
0,1950-01-01,DJF,-1.53,La Nina
1,1950-02-01,JFM,-1.34,La Nina
2,1950-03-01,FMA,-1.16,La Nina
3,1950-04-01,MAM,-1.18,La Nina
4,1950-05-01,AMJ,-1.07,La Nina


In [165]:
# Sort flash flood data chronologically, then check results
flood_df = flood_df.sort_values("begin_date").reset_index(drop=True)

flood_df[["event_id", "begin_date"]].head()

,event_id,begin_date
0,564702,2015-04-03
1,564703,2015-04-03
2,564704,2015-04-03
3,564706,2015-04-03
4,564705,2015-04-03


In [167]:
# Check the data types of the columns before merging into one dataframe
print("flood_df begin_date:", flood_df["begin_date"].dtype)
print("oni_df date:", oni_df["date"].dtype)

flood_df begin_date: str
oni_df date: datetime64[us]


In [169]:
# Change flood_df begin_date from str datatype to datetime datatype
flood_df["begin_date"] = pd.to_datetime(
    flood_df["begin_date"]
)

print(flood_df["begin_date"].dtype)

datetime64[us]


In [171]:
# Merge each flash flood event with the most recent ONI record occurring on or before the event date, then check results
enriched_df = pd.merge_asof(
    flood_df,
    oni_df,
    left_on="begin_date",
    right_on="date",
    direction="backward"
)

enriched_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,injuries_indirect,...,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours,date,oni_season,oni_anomaly,enso_phase
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,0,0,0,0,0,...,-85.6600,38.1536,-85.6547,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,0,0,20000,0,0,...,-85.7000,38.1633,-85.6969,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0,0,30000,0,0,...,-84.8866,38.0243,-84.8836,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0,0,30000,0,0,...,-85.3742,38.1038,-85.3727,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0,0,0,0,0,...,-85.7800,38.2296,-85.7792,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino


In [173]:
# Keep event_id and the ONI columns
enriched_df = enriched_df[
    [
        "event_id",
        "date",
        "oni_season",
        "oni_anomaly",
        "enso_phase"
    ]
]

enriched_df.head()

,event_id,date,oni_season,oni_anomaly,enso_phase
0,564702,2015-04-01,MAM,0.81,El Nino
1,564703,2015-04-01,MAM,0.81,El Nino
2,564704,2015-04-01,MAM,0.81,El Nino
3,564706,2015-04-01,MAM,0.81,El Nino
4,564705,2015-04-01,MAM,0.81,El Nino


In [175]:
# Save the merged DataFrame as a CSV file
enriched_df.to_csv("../data/processed/flash_floods_ky_oni_data.csv", index=False)

## Script #4: NLCD Landcover API

In [ ]:
# Import Libraries and Dependencies 
import os
from pathlib import Path
import pandas as pd
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

In [ ]:
# Load environment variables
load_dotenv()

True

In [178]:
# Add configuration
API_KEY = os.environ.get("ARCGIS_API_KEY")
ARCGIS_URL = "https://arcgis.com"

In [179]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,injuries_indirect,...,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,0,0,0,0,0,...,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,0,0,20000,0,0,...,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0,0,30000,0,0,...,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0,0,30000,0,0,...,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0,0,0,0,0,...,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [ ]:
# Define NLCD Layer IDs
LAYER_IDS = {
    "nlcd": "32e2ccc6416746a9a72b4d216813f84f",
    "elev": "58a541efc59545e6b7137f961d7de883",
    "imperv": "6df535f263dd44f489365eed49461a38",
}

In [181]:
# Define NLCD classes
NLCD_CLASSES = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

In [182]:
# Connect to ArcGIS
def initialize_layers(api_key: str) -> tuple:
    """Connects to the ArcGIS GIS API with strict SSL validation
    and returns the three imagery layers."""

    print("Connecting to ArcGIS Living Atlas layers...")

    gis = GIS(
        ARCGIS_URL,
        api_key=api_key,
        verify_cert=True
    )

    layers = {
        name: ImageryLayer(
            gis.content.get(item_id).url,
            gis=gis
        )
        for name, item_id in LAYER_IDS.items()
    }

    print("All layers connected successfully.")

    return (
        layers["nlcd"],
        layers["elev"],
        layers["imperv"]
    )

In [183]:
# Query One Imagery Layer
def query_layer_value(layer: ImageryLayer, geom: dict):
    """Queries an imagery layer at a point geometry
    and returns its pixel value, or None on failure."""

    try:
        response = layer.identify(
            geometry=geom,
            return_pixel_values=True
        )

        return response.get("value", None)

    except Exception:
        return None

In [184]:
# Process One Flash Flood Event
def process_single_row(idx, row, layers):
    """Worker function to process a single row's
    spatial variables concurrently."""

    nlcd_layer, elev_layer, imp_layer = layers

    mid_lat = (row["begin_lat"] + row["end_lat"]) / 2
    mid_lon = (row["begin_lon"] + row["end_lon"]) / 2

    if pd.isna(mid_lat) or pd.isna(mid_lon):
        return idx, (None, None, None, None)

    geom = {
        "x": mid_lon,
        "y": mid_lat,
        "spatialReference": {"wkid": 4326}
    }

    val_nlcd = query_layer_value(nlcd_layer, geom)
    val_elev = query_layer_value(elev_layer, geom)
    val_imp = query_layer_value(imp_layer, geom)

    c_code = int(val_nlcd) if val_nlcd is not None else None

    c_class = (
        NLCD_CLASSES.get(c_code, "Unknown")
        if c_code is not None
        else None
    )

    elevation = (
        float(val_elev)
        if val_elev is not None
        else None
    )

    impervious = (
        int(val_imp)
        if val_imp is not None
        else None
    )

    return idx, (
        c_code,
        c_class,
        elevation,
        impervious
    )

In [ ]:
# Process all flash flooding events within the dataframe
def fetch_geospatial_attributes(
    df: pd.DataFrame,
    layers: tuple,
    max_workers: int = 20,
) -> pd.DataFrame:

    """Fetches geospatial attributes for each row
    and joins them onto the dataframe."""

    print(
        f"Starting batch execution using "
        f"{max_workers} concurrent threads..."
    )

    cols = [
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]

    results = {}

    total_rows = len(df)

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {
            executor.submit(
                process_single_row,
                idx,
                row,
                layers
            ): idx

            for idx, row in df.iterrows()
        }

        for completed, future in enumerate(
            as_completed(futures), 1
        ):

            idx, values = future.result()

            results[idx] = values

            if completed % 100 == 0 or completed == total_rows:
                print(
                    f"Progress: {completed}/{total_rows} "
                    f"rows extracted "
                    f"({completed/total_rows*100:.1f}%)"
                )

    result_df = pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=cols
    )

    return df.join(result_df)

In [188]:
# Connect to ArcGIS
atlas_layers = initialize_layers(API_KEY)

Connecting to ArcGIS Living Atlas layers...
All layers connected successfully.


In [189]:
# Run geospatial extraction
processed_df = fetch_geospatial_attributes(
    flood_df,
    atlas_layers,
    max_workers=25,
    checkpoint_every=50
)

Starting batch execution using 25 concurrent threads...


c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Progress: 100/1757 rows extracted (5.7%)


c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Progress: 200/1757 rows extracted (11.4%)


...

Progress: 300/1757 rows extracted (17.1%)


c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Progress: 400/1757 rows extracted (22.8%)


c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\msash\Projects\flash_floods_ky_2015_2025\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'di-nlcd.img.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Progress: 500/1757 rows extracted (28.5%)
Progress: 600/1757 rows extracted (34.1%)
Progress: 700/1757 rows extracted (39.8%)
Progress: 800/1757 rows extracted (45.5%)
Progress: 900/1757 rows extracted (51.2%)
Progress: 1000/1757 rows extracted (56.9%)
Progress: 1100/1757 rows extracted (62.6%)
Progress: 1200/1757 rows extracted (68.3%)
Progress: 1300/1757 rows extracted (74.0%)
Progress: 1400/1757 rows extracted (79.7%)
Progress: 1500/1757 rows extracted (85.4%)
Progress: 1600/1757 rows extracted (91.1%)
Progress: 1700/1757 rows extracted (96.8%)
Progress: 1757/1757 rows extracted (100.0%)


In [190]:
# Inspect processed dataset
processed_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,injuries_indirect,...,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours,nlcd_code,nlcd_class,elevation_m,impervious_surface_pct
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,0,0,0,0,0,...,-85.6600,38.1536,-85.6547,2015,0.0,0.0,24,"Developed, High Intensity",149.975,83
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,0,0,20000,0,0,...,-85.7000,38.1633,-85.6969,2015,0.0,0.0,24,"Developed, High Intensity",140.396,84
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0,0,30000,0,0,...,-84.8866,38.0243,-84.8836,2015,0.0,0.0,21,"Developed, Open Space",236.400,0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0,0,30000,0,0,...,-85.3742,38.1038,-85.3727,2015,0.0,0.0,21,"Developed, Open Space",209.800,0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0,0,0,0,0,...,-85.7800,38.2296,-85.7792,2015,0.0,0.0,24,"Developed, High Intensity",142.241,88


In [193]:
# Keep only event_id and NLCD columns
processed_df = processed_df[
    [
        "event_id",
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]
]

processed_df.head()

,event_id,nlcd_code,nlcd_class,elevation_m,impervious_surface_pct
0,564702,24,"Developed, High Intensity",149.975,83
1,564703,24,"Developed, High Intensity",140.396,84
2,564704,21,"Developed, Open Space",236.400,0
3,564706,21,"Developed, Open Space",209.800,0
4,564705,24,"Developed, High Intensity",142.241,88


In [194]:
# Save processed dataset as CSV file
processed_df.to_csv("../data/processed/flash_floods_ky_nlcd_data.csv", index=False)